<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [1]:
import pandas as pd
import plotly.express as px
import os
import geopandas as gpd
import json

# Read data

In [2]:
location = pd.read_csv('data/taxi-zone-lookup.csv')
print('Lines / Columns : ', location.shape)

Lines / Columns :  (265, 3)


In [3]:
files = ["data/uber-raw-data-apr14.csv", "data/uber-raw-data-may14.csv", "data/uber-raw-data-jun14.csv", 'data/uber-raw-data-jul14.csv', 'data/uber-raw-data-aug14.csv', 'data/uber-raw-data-sep14.csv']

raw_2014 = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print('Lines / Columns : ', raw_2014.shape)

Lines / Columns :  (4534327, 4)


In [4]:
raw_2015 = pd.read_csv('data/uber-raw-data-janjune-15.csv')
print('Lines / Columns : ', raw_2015.shape)

Lines / Columns :  (14270479, 4)


In [5]:
# Lowercase dataframes
dataframes = {'location':location, 'raw_2014':raw_2014, 'raw_2015':raw_2015}
for name, df in dataframes.items():
    df.columns = df.columns.str.lower() # lowercase columns names
    for col in df.select_dtypes(include=["object", 'string']).columns: # avoid warning for multi execution
        df[col] = df[col].str.lower()
        print()
    print('-------------------',name,'-------------------')
    print(df.head(1))



------------------- location -------------------
   locationid borough            zone
0           1     ewr  newark airport


------------------- raw_2014 -------------------
          date/time     lat      lon    base
0  4/1/2014 0:11:00  40.769 -73.9549  b02512



------------------- raw_2015 -------------------
  dispatching_base_num          pickup_date affiliated_base_num  locationid
0               b02617  2015-05-17 09:47:00              b02617         141


In [6]:
# os.makedirs('./c') # create directory then deactivate

In [7]:
location.to_csv("c/location.csv", index=False)

# Merge raw_2015 and location

In [8]:
raw_2015 = raw_2015.merge(location[['locationid', "borough", "zone"]], left_on="locationid", right_on="locationid", how="left")
raw_2015.head(1)

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone
0,b02617,2015-05-17 09:47:00,b02617,141,manhattan,lenox hill west


# Treat missing values

La référence du taxi est inutile pour kmeans. On remplace par une étiquette explicite.

In [9]:
raw_2015['affiliated_base_num'] = raw_2015['affiliated_base_num'].fillna('unknown') # drop lines where values are missing. 
missing_values = raw_2015.isnull().sum()/raw_2015.shape[0]*100
print(missing_values)

dispatching_base_num    0.0
pickup_date             0.0
affiliated_base_num     0.0
locationid              0.0
borough                 0.0
zone                    0.0
dtype: float64


In [10]:
import json
import os
import urllib.request

url = "https://data.cityofnewyork.us/api/geospatial/8meu-9t5y?method=export&format=GeoJSON"
urllib.request.urlretrieve(url, "c/taxi_zones.geojson")
print(os.path.abspath("c/taxi_zones.geojson"))

c:\Users\LN\Documents\Jedha\Projets\UBER_PICKUPS\c\taxi_zones.geojson


In [11]:
import json
with open("c/taxi_zones.geojson") as f:
    geojson = json.load(f)
print(geojson["features"][0]["properties"])

{'shape_area': '0.0007823067885', 'locationid': '1', 'shape_leng': '0.116357453189', 'zone': 'Newark Airport', 'borough': 'EWR'}


In [12]:
len(raw_2015)

14270479

In [13]:
# ajout des colonnes lon, lat sur raw_2015
zones = gpd.read_file("c/taxi_zones.geojson").dissolve(by="locationid").reset_index()
centroids = zones.to_crs(epsg=2263).geometry.centroid.to_crs(epsg=4326)
zones["lon"], zones["lat"] = centroids.x, centroids.y
zones["locationid"] = zones["locationid"].astype("int64")
zones = zones[["locationid", "lon", "lat"]]

len(raw_2015)

14270479

In [14]:
raw_2015 = raw_2015.merge(zones, on="locationid", how="left")

# Merge pickup_14 et taxi_zones.geojson

In [15]:
before = len(raw_2014)

In [16]:
zones = gpd.read_file("c/taxi_zones.geojson").to_crs(epsg=4326)[["locationid", "geometry"]]
zones["locationid"] = zones["locationid"].astype("int64")

points = gpd.GeoDataFrame(raw_2014, geometry=gpd.points_from_xy(raw_2014["lon"], raw_2014["lat"]), crs="EPSG:4326")
raw_2014 = pd.DataFrame(gpd.sjoin(points, zones, how="left", predicate="within").drop(columns=["geometry", "index_right"]))
after = len(raw_2014)

In [17]:
if before - after != 0:
    print('Verify rows number')
if raw_2014.shape[1] != 5:
    print('Verify columns number')
    raw_2014.head(1)

In [18]:
raw_2014.head(1)

,date/time,lat,lon,base,locationid
0,4/1/2014 0:11:00,40.769,-73.9549,b02512,140.0


In [19]:
# Merge raw_2015 and location
raw_2014 = raw_2014.merge(location[['locationid', "borough", "zone"]], left_on="locationid", right_on="locationid", how="left")
raw_2014.head(1)

,date/time,lat,lon,base,locationid,borough,zone
0,4/1/2014 0:11:00,40.769,-73.9549,b02512,140.0,manhattan,lenox hill east


# Treat datetime

In [20]:
cols = [raw_2014["date/time"], raw_2015["pickup_date"]]

converted = [pd.to_datetime(c, format=f) for c, f in zip(cols, ("%m/%d/%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S")) # zip to use c and f together
]

raw_2014["date/time"], raw_2015["pickup_date"] = converted
print('raw_2014 : ', raw_2014["date/time"].dtypes, 'raw_2015 : ', raw_2015["pickup_date"].dtypes) # datetime64[ns]

# print([c.dtype for c in converted])  # # M8[ns]

raw_2014 :  datetime64[ns] raw_2015 :  datetime64[ns]


In [21]:
pickup_14 = raw_2014
pickup_14.to_csv("c/2014.csv", index=False)

# Analyze one hour on one day

In [22]:
pickup = raw_2015
pickup.to_csv("c/2015.csv", index=False)
pickup.head(1)

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone,lon,lat
0,b02617,2015-05-17 09:47:00,b02617,141,manhattan,lenox hill west,-73.959635,40.766948


# Merge pickup et pickup_14

In [25]:
pickup_14 = pickup_14.rename(columns={"date/time": "pickup_date", "base": "dispatching_base_num"})
df = pd.concat([pickup_14, pickup], ignore_index=True)
pickup.head(1)

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone,lon,lat
0,b02617,2015-05-17 09:47:00,b02617,141,manhattan,lenox hill west,-73.959635,40.766948


In [26]:
if len(df) == len(pickup_14) + len(pickup) and df.shape[1] == 8:
    print("Complete data")
elif df.shape[1] != 8:
    print('Different rows')
    df.head(1)
else: 
    print("Missing lines")

Complete data


In [27]:
df.head(1)

,pickup_date,lat,lon,dispatching_base_num,locationid,borough,zone,affiliated_base_num
0,2014-04-01 00:11:00,40.769,-73.9549,b02512,140.0,manhattan,lenox hill east,NaN


In [ ]:
for col in df.columns:
    print(df[col].map(type).value_counts().to_string(index=True))

pickup_date
<class 'pandas._libs.tslibs.timestamps.Timestamp'>    18713730
lat
<class 'float'>    18713730
lon
<class 'float'>    18713730
dispatching_base_num
<class 'str'>    18713730
locationid
<class 'int'>    18713730
borough
<class 'str'>    18713730
zone
<class 'str'>    18713730
affiliated_base_num
<class 'str'>    18713730


In [29]:
print((df['locationid'].isna().mean() * 100).round(2))

0.48


In [30]:
df = df.dropna(subset=['locationid'])
df['locationid'] = df['locationid'].astype(int)

In [31]:
cols = ['dispatching_base_num', 'affiliated_base_num', 'borough', 'zone']
df[cols] = df[cols].astype(str)


In [32]:
df.to_csv("c/df.csv", index=False)